# ScoreIA - Documento Tecnico do Pipeline de ML

Especificacao operacional do pipeline de modelagem.
Snapshot tecnico atualizado em: `2026-03-09`.


## Fluxo de Execucao

1. Capturar parametros de execucao na interface web.
2. Executar mini-pipeline opcional de validacao do dataset, inclusive validacao temporal/OOT.
3. Criar `run_context` com timestamp unico da execucao.
4. Carregar dados (`csv`/`parquet`) e separar `train`, `test` e `oot` antes de qualquer fit.
5. Ajustar preprocessing apenas no treino.
6. Executar feature selection temporal por estabilidade, quando habilitado.
7. Executar `model_full`:
   - CV,
   - Optuna opcional,
   - treino final,
   - threshold de classificacao,
   - metricas,
   - interpretabilidade,
   - relatorio,
   - exportacao de artefatos.
8. Executar `model_prunned`, quando habilitado:
   - derivar subconjunto por corte acumulado de importancia no `model_full`,
   - reexecutar CV/Optuna,
   - retreinar modelo final,
   - regenerar relatorio e artefatos do estagio.
9. Executar `model_reduced`, quando habilitado:
   - usar `model_prunned` como origem, se existir; caso contrario usar `model_full`,
   - selecionar as `Y` features menos importantes do estagio de origem,
   - aplicar PCA na cauda,
   - manter o menor numero de componentes que explica pelo menos `70%` da variabilidade,
   - reexecutar CV/Optuna,
   - retreinar modelo final,
   - regenerar relatorio e artefatos do estagio.
10. Registrar execucao na memoria do agente e retornar o estagio final efetivo.


## Parametros de Entrada (Interface Web)

- `file`: arquivo de dados para treino.
- `target`: variavel resposta.
- `task`: `regression` ou `classification`.
- `formato`: `csv` ou `parquet`.
- `modelo`: `lightgbm`, `mlp` ou `auto`.
- `otimizar`: habilita/desabilita Optuna.
- `n_trials`: numero de tentativas da otimizacao.
- `early_stopping_rounds`: paciencia do early stopping no CV/Optuna.
- `apply_missing_treatment`: liga/desliga imputacao no preprocessing.
- `temporal_feature`: coluna temporal em formato `YYYYMMDD` ou `YYYYMM`.
- `oot_months`: numero de meses mais recentes reservados para OOT.
- `segment_features`: lista dinamica de variaveis de segmentacao.
- `segment_strategy`: `global`, `bucket_feature` ou `per_bucket_model`.
- `stability_feature_selection`: ativa selecao temporal por estabilidade.
- `stability_max_folds`: numero maximo de folds temporais.
- `stability_max_candidates`: numero maximo de candidatos de tamanho de conjunto de features.
- `stability_perf_rel_tolerance`: tolerancia relativa de performance entre folds.
- `stability_perf_abs_tolerance`: tolerancia absoluta adicional.
- `stability_importance_cutoff_pct`: corte acumulado de importancia dentro do feature selection temporal.
- `prune_best_model`: ativa o estagio `model_prunned`.
- `prune_importance_cutoff_pct`: percentual acumulado de importancia usado no pruning do melhor modelo.
- `reduce_tail_features`: ativa o estagio `model_reduced`.
- `reduction_tail_features_count`: numero `Y` de variaveis menos importantes que entram no PCA.
- `reduction_variance_threshold`: threshold de variancia explicada para o PCA da cauda (default atual: `0.70`).
- `classification_balance_strategy`: `none`, `class_weight`, `undersample` ou `oversample`.
- `positive_class_weight`: peso explicito da classe positiva quando a estrategia e `class_weight`.


## Regras de Modelagem

- Sem leakage: `test` e `oot` permanecem intocados ate a avaliacao.
- CV acontece somente no conjunto de treino do estagio corrente.
- Metrica primaria por task:
  - Regressao: `RMSE` (`minimize`).
  - Classificacao: `AUC` (`maximize`).
- Para classificacao:
  - Optuna/CV operam com probabilidades para `AUC`.
  - O threshold final e otimizado por precision-recall.
  - O mesmo threshold e reutilizado em `train`, `test`, `oot`, scored dataset e interpretabilidade.
- `temporal_feature` nunca entra como preditora no ajuste.
- O estagio `model_prunned` somente existe se o corte acumulado realmente reduzir o conjunto do estagio de origem.
- O estagio `model_reduced` somente existe se houver cauda suficiente para PCA util.
- O estagio final efetivo pode ser `model_full`, `model_prunned` ou `model_reduced`.


## Multiestagio de Modelagem

### `model_full`
- Primeiro modelo final do pipeline apos preprocessing e feature selection temporal.
- Serve como baseline tecnico da execucao.

### `model_prunned`
- Usa o ranking de importancia do `model_full`.
- Seleciona o menor conjunto de features que acumula `X%` da importancia definida pelo usuario.
- Reexecuta Optuna/CV e retreina o modelo final apenas com esse subconjunto.

### `model_reduced`
- Usa `model_prunned` como origem quando ele existe; caso contrario usa `model_full`.
- Ordena features por importancia do estagio de origem.
- Mantem a cabeca mais importante e substitui a cauda de `Y` variaveis por componentes principais.
- O numero de componentes e o menor que explica ao menos `70%` da variabilidade da cauda.
- Reexecuta Optuna/CV e retreina o modelo final com `head + componentes PCA`.


## Interpretabilidade por Tecnica

- LightGBM:
  - feature importance do modelo,
  - SHAP top features,
  - scatter SHAP vs dominio da feature,
  - identificacao de possivel inversao de efeito.
- MLP:
  - permutation importance,
  - ALE 1D para top features,
  - leitura de direcao marginal por dominio.
- Cada estagio gerado tem sua propria interpretabilidade exportada.


## Relatorio Tecnico HTML - Secoes Obrigatorias

- Contexto da execucao e parametros de entrada.
- Volumetria total e splits `train` / `test` / `oot`.
- Metadata do pipeline e do estagio.
- Resumo de otimizacao e grafico do processo Optuna.
- Curva de aprendizado do estagio.
- Metricas de `train`, `test` e `oot`.
- Top 20 importancias.
- SHAP ou ALE, conforme a tecnica.
- Analise temporal/segmentativa.
- Grafico temporal combinado:
  - barras para volume,
  - linhas para media do target,
  - eixo duplo.
- Resumo do feature selection temporal por estabilidade.


## Modulos Principais do Pipeline

- `core/agent.py`: orquestracao de alto nivel e consolidacao do retorno.
- `tools/ml_pipeline_flow.py`: executor do pipeline multiestagio.
- `tools/data_pipeline.py`: split, holdout, validacao temporal e preprocessing.
- `tools/feature_selection.py`: selecao temporal por estabilidade.
- `tools/feature_postprocessing.py`: encadeamento de selecao de features e PCA da cauda.
- `tools/model_training.py`: treino final, predicao, importance profile e bucket ensemble.
- `tools/optimization.py`: Optuna/CV.
- `tools/interpretability.py`: SHAP, permutation importance e ALE.
- `tools/reporting.py`: HTML e graficos SVG.
- `tools/artifact_export.py`: pacote reprodutivel por estagio.
- `tools/webapp/app.py`: interface web.
- `tools/webapp/validation.py`: mini-pipeline de validacao.
- `tools/webapp/storage.py`: persistencia temporaria do dataset enviado via web.


## Arvore de Artefatos

Estrutura por execucao:

```text
artifacts/
  run_<timestamp>/
    model_full/
      model/
      preprocess/
      inference/
      metadata/
      reports/
      scored_data/
      requirements_<timestamp>.txt
      pyproject_<timestamp>.toml
      poetry_<timestamp>.lock
    model_prunned/
      ... mesma estrutura ...
    model_reduced/
      ... mesma estrutura ...
```

Cada pasta de estagio pode conter:
- `model/model_<timestamp>.joblib` ou `.pt`
- `preprocess/pipeline_<timestamp>.joblib`
- `inference/predict_<timestamp>.py`
- `metadata/model_metadata_<timestamp>.json`
- `metadata/train_config_<timestamp>.json`
- `metadata/metrics_train_<timestamp>.json`
- `metadata/metrics_test_<timestamp>.json`
- `metadata/feature_schema_<timestamp>.json`
- `metadata/optuna_summary_<timestamp>.json`
- `metadata/versions_<timestamp>.json`
- `metadata/artifact_hashes_<timestamp>.json`
- `metadata/artifact_manifest_<timestamp>.json`
- `reports/report_<timestamp>.html`
- `reports/interpretability_<timestamp>.json`
- `scored_data/scored_dataset_<timestamp>.csv`


## Contrato de Saida Consolidado

O retorno do pipeline inclui:
- `final_stage_name`
- `modelo_final`
- `metricas_treino`, `metricas`, `metricas_oot`
- `threshold_classificacao` (quando aplicavel)
- `selected_feature_names`
- `feature_selection_stability`
- `pruning_summary`
- `reduction_summary`
- `caminho_relatorio_html`
- `caminho_interpretabilidade`
- `caminho_dataset_scored`
- `model_stage_outputs` com os caminhos e resumo de cada estagio realmente executado.
